In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/telco_cleaned.csv')

print("Shape:", df.shape)
df.head()

Shape: (7043, 38)


,Gender,Age,Senior Citizen,Married,Dependents,Number of Dependents,Population,Referred a Friend,Number of Referrals,Tenure in Months,...,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Churn Label,CLTV,Churn Category,Churn Reason
0,Male,78,Yes,No,No,0,68701,No,0,1,...,39.65,39.65,0.00,20,0.00,59.65,Yes,5433,Competitor,Competitor offered more data
1,Female,74,Yes,Yes,Yes,1,55668,Yes,1,8,...,80.65,633.30,0.00,0,390.80,1024.10,Yes,5302,Competitor,Competitor made better offer
2,Male,71,Yes,No,Yes,3,47534,No,0,18,...,95.45,1752.55,45.61,0,203.94,1910.88,Yes,3179,Competitor,Competitor made better offer
3,Female,78,Yes,Yes,Yes,1,27778,Yes,1,25,...,98.50,2514.50,13.43,0,494.00,2995.07,Yes,5337,Dissatisfaction,Limited range of services
4,Female,80,Yes,Yes,Yes,1,26265,Yes,1,37,...,76.50,2868.15,0.00,0,234.21,3102.36,Yes,2793,Price,Extra data charges


In [2]:
# Target variable
y = df['Churn Label']

# Features - Churn Category aur Churn Reason ko hatao (RAG ke liye reserve)
# Churn Label bhi hatao (yeh target hai, feature nahi)
X = df.drop(columns=['Churn Label', 'Churn Category', 'Churn Reason'])

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

Features shape: (7043, 35)
Target shape: (7043,)

Feature columns:
['Gender', 'Age', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Population', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'CLTV']


In [3]:
# Target ko encode karo: Yes -> 1, No -> 0
y = y.map({'Yes': 1, 'No': 0})

print(y.value_counts())
print("\nData type:", y.dtype)

Churn Label
0    5174
1    1869
Name: count, dtype: int64

Data type: int64


In [4]:
# Categorical aur numerical columns alag karo
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nNumerical columns:", len(numerical_cols))
print(numerical_cols)

Categorical columns: 21
['Gender', 'Senior Citizen', 'Married', 'Dependents', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical columns: 14
['Age', 'Number of Dependents', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'CLTV']


In [5]:
# One-Hot Encoding - categorical columns ko numeric mein convert karo
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("Original shape:", X.shape)
print("After encoding:", X_encoded.shape)

Original shape: (7043, 35)
After encoding: (7043, 43)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.2,        # 20% data testing ke liye
    random_state=42,      # reproducibility ke liye (same split har baar)
    stratify=y             # churn ratio dono sets mein same rahe
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining churn ratio:\n", y_train.value_counts(normalize=True))
print("\nTesting churn ratio:\n", y_test.value_counts(normalize=True))

Training set: (5634, 43)
Testing set: (1409, 43)

Training churn ratio:
 Churn Label
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Testing churn ratio:
 Churn Label
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Feature scaling - Logistic Regression ke liye zaroori hai
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model train karo
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predictions karo test set pe
y_pred = log_reg.predict(X_test_scaled)

# Metrics calculate karo
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nFull Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8502484031227822
Precision: 0.7404129793510325
Recall: 0.6711229946524064
F1-Score: 0.7040673211781207

Confusion Matrix:
[[947  88]
 [123 251]]

Full Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.91      0.90      1035
           1       0.74      0.67      0.70       374

    accuracy                           0.85      1409
   macro avg       0.81      0.79      0.80      1409
weighted avg       0.85      0.85      0.85      1409



In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1-Score:", f1_score(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.8417317246273953
Precision: 0.7612456747404844
Recall: 0.5882352941176471
F1-Score: 0.6636500754147813

Confusion Matrix:
[[966  69]
 [154 220]]


In [10]:
from xgboost import XGBClassifier

# scale_pos_weight - imbalanced data handle karne ke liye
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=200, 
    random_state=42, 
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1-Score:", f1_score(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

Accuracy: 0.8346344925479063
Precision: 0.6831168831168831
Recall: 0.7032085561497327
F1-Score: 0.6930171277997365

Confusion Matrix:
[[913 122]
 [111 263]]


In [11]:
# Probability scores nikalo (0.5 fixed threshold ke bajaye)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Different thresholds try karo
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5]:
    y_pred_thresh = (y_pred_proba >= threshold).astype(int)
    prec = precision_score(y_test, y_pred_thresh)
    rec = recall_score(y_test, y_pred_thresh)
    f1 = f1_score(y_test, y_pred_thresh)
    print(f"Threshold: {threshold} | Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f}")

Threshold: 0.3 | Precision: 0.638 | Recall: 0.783 | F1: 0.703
Threshold: 0.35 | Precision: 0.651 | Recall: 0.767 | F1: 0.704
Threshold: 0.4 | Precision: 0.656 | Recall: 0.738 | F1: 0.694
Threshold: 0.45 | Precision: 0.664 | Recall: 0.714 | F1: 0.688
Threshold: 0.5 | Precision: 0.683 | Recall: 0.703 | F1: 0.693


In [12]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

xgb_base = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, eval_metric='logloss')

random_search = RandomizedSearchCV(
    xgb_base, 
    param_distributions=param_grid, 
    n_iter=30,             # 30 combinations try karega
    scoring='recall',       # Recall ko optimize karega (humara priority metric)
    cv=3,                    # 3-fold cross validation
    random_state=42,
    n_jobs=-1                # sab CPU cores use karega, fast hoga
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV Recall score:", random_search.best_score_)

Best parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
Best CV Recall score: 0.878318634881264


In [13]:
# Best model nikalo
best_xgb = random_search.best_estimator_

# Test set pe predict karo
y_pred_best = best_xgb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("Recall:", recall_score(y_test, y_pred_best))
print("F1-Score:", f1_score(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

Accuracy: 0.7714691270404542
Precision: 0.5427631578947368
Recall: 0.8823529411764706
F1-Score: 0.6720977596741344

Confusion Matrix:
[[757 278]
 [ 44 330]]


In [14]:
pip install imbalanced-learn

In [15]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_smote.value_counts().to_dict())

xgb_smote = XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
xgb_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = xgb_smote.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall:", recall_score(y_test, y_pred_smote))
print("F1-Score:", f1_score(y_test, y_pred_smote))

Before SMOTE: {0: 4139, 1: 1495}
After SMOTE: {0: 4139, 1: 4139}

Accuracy: 0.8346344925479063
Precision: 0.7008547008547008
Recall: 0.6577540106951871
F1-Score: 0.6786206896551724


In [16]:
# Tuned model ki probabilities nikalo
y_pred_proba_tuned = best_xgb.predict_proba(X_test)[:, 1]

# Different thresholds try karo tuned model pe
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_pred_proba_tuned >= threshold).astype(int)
    prec = precision_score(y_test, y_pred_t)
    rec = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold: {threshold} | Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f}")

Threshold: 0.3 | Precision: 0.438 | Recall: 0.987 | F1: 0.607
Threshold: 0.4 | Precision: 0.497 | Recall: 0.949 | F1: 0.652
Threshold: 0.5 | Precision: 0.543 | Recall: 0.882 | F1: 0.672
Threshold: 0.6 | Precision: 0.631 | Recall: 0.762 | F1: 0.690
Threshold: 0.7 | Precision: 0.753 | Recall: 0.537 | F1: 0.627


In [17]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Final model (default XGBoost) save karo
joblib.dump(xgb_model, '../models/churn_model.pkl')

# Best threshold bhi save karo (Streamlit app mein use hoga)
best_threshold = 0.3
joblib.dump(best_threshold, '../models/best_threshold.pkl')

# Feature columns save karo (encoding ke baad wale)
joblib.dump(X_encoded.columns.tolist(), '../models/feature_columns.pkl')

# Scaler bhi save kar lo (agar kabhi Logistic Regression use karna pade)
joblib.dump(scaler, '../models/scaler.pkl')

print("Model, threshold, feature columns, aur scaler — sab saved successfully!")

Model, threshold, feature columns, aur scaler — sab saved successfully!
